# Amber Advanced Tutorial : Placeholder
## Preparation of the SUSTAINER Residue

This notebook automates Section 2 of the [Amber Tutorial 3](https://ambermd.org/tutorials/basic/tutorial15/index.php). 
It is configured to run AmberTools commands inside an **Apptainer** container (.sif).

### Prerequisites
* Ensure **Apptainer** is installed.
* Update the `SIF_PATH` below to point to the project container file.

In [1]:
import os
import subprocess

# --- CONFIGURE THIS PATH ---
SIF_PATH = "amber_ready.sif" 

def run_apptainer(cmd):
    full_cmd = f"apptainer exec {SIF_PATH} {cmd}"
    print(f"Executing: {full_cmd}")
    
    # Use subprocess to capture errors
    result = subprocess.run(full_cmd, shell=True, capture_output=True, text=True)
    
    if result.returncode != 0:
        print("❌ COMMAND FAILED!")
        print("STDOUT:", result.stdout)
        print("STDERR:", result.stderr)
    else:
        print("✅ Success!")
        print(result.stdout)

# Verify the container exists
if not os.path.exists(SIF_PATH):
    print(f"Warning: Container not found at {SIF_PATH}. Please update the path.")

In [ ]:
run_apptainer("which pdb4amber")
run_apptainer("which tleap")
run_apptainer("which sander")



In [3]:
subprocess.run("wget -q https://files.rcsb.org/download/1L2Y.pdb", shell=True)

CompletedProcess(args='wget -q https://files.rcsb.org/download/1L2Y.pdb', returncode=0)

In [ ]:
run_apptainer("pdb4amber -i 1L2Y.pdb -o 1l2y.amber.pdb")

In [5]:
with open("tleap.in", "w") as f:
    f.write("""
source leaprc.protein.ff14SBonlysc
tc5b = loadpdb 1l2y.amber.pdb
set default PBradii mbondi3
saveamberparm tc5b tc5b.1l2y.parm7 tc5b.1l2y.rst7
quit
""")

In [6]:
run_apptainer("tleap -f tleap.in")

Executing: apptainer exec amber_ready.sif tleap -f tleap.in
❌ COMMAND FAILED!
STDOUT: 
STDERR: /bin/sh: apptainer: command not found



In [7]:
with open("min.in", "w") as f:
    f.write("""
Minimization
 &cntrl
  imin=1,
  maxcyc=50,
  ntx=1,
  ntwr=50,
  ntpr=10,
  ntb=0,
  igb=8,
 /
""")

In [ ]:
run_apptainer("sander -O -i min.in -p tc5b.1l2y.parm7 -c tc5b.1l2y.rst7 -o min.out -r min.rst7")

In [ ]:
run_apptainer("tail -30 min.out")